# Testing Pandas for EBD Processing

Note: this data is pulling from an unzipped txt EBD file in the "ebd" folder not included in the repository

## Resources

- [Pandas vs R Cheatsheet](https://datascientyst.com/pandas-vs-r-cheat-sheet/)
- 

In [28]:
import pandas as pd
import json


# eBird Dataset file name
ebd_fn = 'ebd_US-NC_202101_202602_unv_smp_relMar-2026.txt'

## Load the EBD into a Pandas DF

In [29]:
# define data types
dtypes = {'LAST EDITED DATE' : str, 'COUNTRY' : str, 'COUNTRY CODE' : str,
'STATE' : str, 'STATE CODE' : str, 'COUNTY' : str, 'COUNTY CODE' : str,
'IBA CODE' : str, 'BCR CODE' : str, 'USFWS CODE' : str, 'ATLAS BLOCK' : str,
'LOCALITY' : str, 'LOCALITY ID' : str, 'LOCALITY TYPE' : str,
'LATITUDE' : float, 'LONGITUDE' : float, 'OBSERVATION DATE' : str,
'TIME OBSERVATIONS STARTED' : str, 'OBSERVER ID' : str, 
'OBSERVER ORCID ID' : str, 'SAMPLING EVENT IDENTIFIER' : str,
'OBSERVATION TYPE' : str, 'PROTOCOL NAME' : str, 'PROTOCOL CODE' : str,
'PROJECT NAMES' : object, 'PROJECT IDENTIFIERS' : object, 'DURATION MINUTES' : float,
'EFFORT DISTANCE KM' : str, 'EFFORT AREA HA' : str, 'NUMBER OBSERVERS' : float,
'ALL SPECIES REPORTED' : str, 'GROUP IDENTIFIER' : str, 'HAS MEDIA' : str,
'CHECKLIST COMMENTS' : str, 'GLOBAL UNIQUE IDENTIFIER' : str, 
'TAXONOMIC ORDER' : str, 'CATEGORY' : str, 'TAXON CONCEPT ID' : str,
'COMMON NAME' : str, 'SCIENTIFIC NAME' : str, 'SUBSPECIES COMMON NAME' : str,
'SUBSPECIES SCIENTIFIC NAME' : str, 'EXOTIC CODE' : str, 
'OBSERVATION COUNT' : str, 'BREEDING CODE' : str, 'BREEDING CATEGORY' : str,
'BEHAVIOR CODE' : str, 'AGE/SEX' : str, 'APPROVED' : str, 'REVIEWED' : str,
'REASON' : str, 'SPECIES COMMENTS' : str}

ebd_raw = pd.read_csv(f'./ebd/{ebd_fn}', sep='\t', dtype = dtypes)

MemoryError: Unable to allocate 7.17 GiB for an array with shape (48, 20052478) and data type object

## Start Exploring

In [21]:
rows_columns = ebd_raw.shape
print(f'{rows_columns[0]:,d} rows, {rows_columns[1]} columns found')
# ebd['SAMPLING EVENT IDENTIFIER'].sort_values()

# remove spaces from colum names
ebd_raw.columns = ebd_raw.columns.str.replace(' ', '_')

# get column names
columns =  ebd_raw.columns

# define observation fields
observation_fields = pd.Series([
  "GLOBAL_UNIQUE_IDENTIFIER",
  "TAXONOMIC_ORDER",
  "CATEGORY",
  "TAXON_CONCEPT_ID",
  "COMMON_NAME",
  "SCIENTIFIC_NAME",
  "SUBSPECIES_COMMON_NAME",
  "SUBSPECIES_SCIENTIFIC_NAME",
  "EXOTIC_CODE",
  "OBSERVATION_COUNT",
  "BREEDING_CODE",
  "BREEDING_CATEGORY",
  "BEHAVIOR_CODE",
  "AGE/SEX",
  "APPROVED",
  "REVIEWED",
  "REASON",
  "SPECIES_COMMENTS"
])

# get checklist fields
checklist_fields = pd.Series(columns[~columns.isin(observation_fields)])

# add SAMPLING EVENT IDENTIFIER to observations, remove Unamed column from checklist_fields
observation_fields = pd.concat([observation_fields, pd.Series(['SAMPLING_EVENT_IDENTIFIER'])])
checklist_fields = checklist_fields[checklist_fields != 'Unnamed: 52']

print(f'checklist fields:\n{checklist_fields}')
print(f'observation fields:\n{observation_fields}')

20,052,478 rows, 53 columns found
checklist fields:
0              LAST_EDITED_DATE
1                       COUNTRY
2                  COUNTRY_CODE
3                         STATE
4                    STATE_CODE
5                        COUNTY
6                   COUNTY_CODE
7                      IBA_CODE
8                      BCR_CODE
9                    USFWS_CODE
10                  ATLAS_BLOCK
11                     LOCALITY
12                  LOCALITY_ID
13                LOCALITY_TYPE
14                     LATITUDE
15                    LONGITUDE
16             OBSERVATION_DATE
17    TIME_OBSERVATIONS_STARTED
18                  OBSERVER_ID
19            OBSERVER_ORCID_ID
20    SAMPLING_EVENT_IDENTIFIER
21             OBSERVATION_TYPE
22                PROTOCOL_NAME
23                PROTOCOL_CODE
24                PROJECT_NAMES
25          PROJECT_IDENTIFIERS
26             DURATION_MINUTES
27           EFFORT_DISTANCE_KM
28               EFFORT_AREA_HA
29             NUMBE

## Nest Observations, create JSON

In [22]:
# create subset for testing
ebd = ebd_raw.head(1000)
ebd = ebd.fillna('')
# ebd = ebd_raw

out_json = open('test.json', 'w', encoding = 'utf-8-sig')


In [27]:
# obs_only = ebd[observation_fields]
# check_only = ebd[checklist_fields]

# print(list(checklist_fields))

# print(ebd.head(10))
# print(ebd.columns)
# ebd_grouped = ebd.groupby('SAMPLING EVENT IDENTIFIER')
# ebd_json = ebd_grouped.apply(
#     lambda x: x.drop('SAMPLING EVENT IDENTIFIER',
#     axis = 1
#     ).to_dict(orient='records')
# ).to_json()

# checklist_field_list = list(checklist_fields)
# print(checklist_field_list)
# ebd.groupby(list(checklist_field_list)).size()

# add fields
ebd['YEAR'] = ebd['OBSERVATION_DATE'].str[:4]
ebd['MONTH'] = ebd['OBSERVATION_DATE'].str[5:7]
ebd['PROJECT_CODE'] = ""
ebd['ATLAS_BLOCK'] = ""
ebd['ID_NCBA_BLOCK'] = ""
ebd['ID_BLOCK_CODE'] = ""
ebd['PRIORITY_BLOCK'] = ""
ebd['NCBA_JULIAN_DAY'] = ""
ebd['NCBA_WEEK'] = ""
ebd['NCBA_SEASON'] = ""
ebd['NCBA_QUARTER'] = ""
ebd['NCBA_DATE_ADDED'] = "2026-04-27"
ebd['NCBA_EBD_VER'] = "Mar-2026"
ebd['NCBA_OBSERVER'] = "Mar-2026"


ebd_json = (
    # ebd.groupby(checklist_fields)
    ebd.groupby([
        'SAMPLING_EVENT_IDENTIFIER', 'LAST_EDITED_DATE', 'COUNTRY',
        'COUNTRY_CODE', 'STATE', 'STATE_CODE', 'COUNTY', 'COUNTY_CODE',
        'IBA_CODE', 'BCR_CODE', 'USFWS_CODE', 'ATLAS_BLOCK', 'LOCALITY',
        'LOCALITY_ID', 'LOCALITY_TYPE', 'LATITUDE', 'LONGITUDE',
        'OBSERVATION_DATE', 'TIME_OBSERVATIONS_STARTED', 'OBSERVER_ID',
        'PROTOCOL_NAME', 'PROTOCOL_CODE', 'PROJECT_CODE', 
        'DURATION_MINUTES', 'EFFORT_DISTANCE_KM', 'EFFORT_AREA_HA',
        'NUMBER_OBSERVERS', 'ALL_SPECIES_REPORTED', 'GROUP_IDENTIFIER',
        'CHECKLIST_COMMENTS', 'YEAR', 'MONTH', 'ID_BLOCK_CODE', 'ID_NCBA_BLOCK',
        'PRIORITY_BLOCK', 'NCBA_JULIAN_DAY', 'NCBA_WEEK', 'NCBA_SEASON',
        'NCBA_DATE_ADDED', 'NCBA_EBD_VER', 'NCBA_OBSERVER'
        ])
    # ebd.groupby(['SAMPLING EVENT IDENTIFIER'])
    .apply(lambda x: x[list(observation_fields)].to_dict('records'))
    .reset_index()
    .rename(columns={0:'OBSERVATIONS'})
    .to_dict(orient='records')
)
out_json.write(json.dumps(ebd_json, indent = 2))


C:\Users\skanderson\AppData\Local\Temp\1\ipykernel_15064\1340600135.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x[list(observation_fields)].to_dict('records'))


1057990

In [24]:
# data = {'productLine': ['Alpha', 'Alpha', 'Beta'],
#         'color': ['Red', 'Blue', 'Green'],
#         'specs': [[{'partId': 'A1', 'weight': 10}, {'partId': 'A2', 'weight': 20}],
#                   [{'partId': 'B1', 'weight': 30}],
#                   [{'partId': 'C1', 'weight': 40}]]}

# df = pd.DataFrame(data)
# # print(json.dumps(df.to_json(orient='records'), indent=2))
# with open('test.json', 'w', encoding = "utf-8-sig") as f:
#     json.dump(df, f, ensure_ascii=False, indent = 2)
